In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
from tqdm import tqdm

# define farfield animation
X_DIM = 256
Y_DIM = 256
Z_DIM = 256
Z_SCALE = 1e-9
TIME_STEPS = 120

TARGET_X_DIM = 32
TARGET_Y_DIM = 32
TARGET_Z_DIM = 32

farfield_zone = np.zeros((TIME_STEPS, X_DIM, Y_DIM, Z_DIM))

In [ ]:
def get_target_path(steps, x_max, y_max, z_max):
    """
    Calculates the (x, y, z) coordinates for a target moving in a 
    horizontal square perimeter at the midpoint of Y.
    """
    path = []
    y_mid = y_max // 2
    
    offset = x_max / 16

    x_min = offset
    x_max_eff = x_max - offset - 1
    z_min = offset
    z_max_eff = z_max - offset - 1

    x_dist = x_max_eff - x_min
    z_dist = z_max_eff - z_min

    # divide total steps into 4 segments for the 4 sides of the square
    steps_per_side = steps // 4
    
    for t in range(steps):
        side = t // steps_per_side
        fraction = (t % steps_per_side) / steps_per_side
        
        if side == 0: # +x movement on front face
            x = x_min + int(fraction * x_dist)
            z = z_min
        elif side == 1: # +z movement on right face
            x = x_max_eff
            z = z_min + int(fraction * z_dist)
        elif side == 2: # -x movement on back face
            x = x_min + int((1 - fraction) * x_dist)
            z = z_max_eff
        else: # -z movement on left face back to starting position
            x = x_min
            z = z_min + int((1 - fraction) * z_dist)
            
        path.append((x, y_mid, z))
        
    return path

In [ ]:
path_coords = get_target_path(TIME_STEPS, X_DIM, Y_DIM, Z_DIM)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# set fixed axis limits - stop potential resizing during target travel
# matplotlib expects (width, height, depth) -> swap y/z
ax.set_xlim(0, X_DIM)
ax.set_zlim(0, Y_DIM)
ax.set_ylim(0, Z_DIM)

# add labels
ax.set_xlabel('X Voxel')
ax.set_ylabel('Z Voxel')
ax.set_zlabel('Y Voxel (Height)')
ax.set_title('3D Target Movement (GIF Generation)')

# draw wireframe at the midpoint to show the path's plane
xx, zz = np.meshgrid([0, X_DIM], [0, Z_DIM])
yy = np.full_like(xx, Y_DIM // 2)
ax.plot_wireframe(xx, zz, yy, color='gray', alpha=0.2)

# initialize the plot element that will move (the target voxel)
# we start with empty data []; 
# s (size)=150, c (color)='red' is color
target_scatter = ax.scatter([], [], [], c='red', s=150, marker='o', edgecolors='black')

def update(frame):
    """
    This function is called for every frame of the animation.
    'frame' is an integer counting from 0 to frames-1.
    """
    # get coordinate for the current time step
    x, y, z = path_coords[frame]
    
    # update the 3D scatter plot position.
    # matplotlib 3D scatter requires updating internal _offsets3d
    # wrap x, y, z in lists because scatter expects sequences
    # Swap z and y axis since - matplotlib expects second value to be depth, last to be height
    target_scatter._offsets3d = ([x], [z], [y])
    
    # update title for each time step
    ax.set_title(f'Target Movement - Time Step: {frame}/{TIME_STEPS}')
    
    # return updated display
    return target_scatter,

def visualize_path(path, x_dim, y_dim, z_dim):
    # extract x, z, y coordinates from the list of tuples
    # path is expected to be [(x1, z1, y1), (x2, z2, y2), ...]
    path_np = np.array(path)
    x_coords = path_np[:, 0]
    y_coords = path_np[:, 1]
    z_coords = path_np[:, 2]

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # plot target path
    ax.plot(x_coords, z_coords, y_coords, color='blue', label='Target Path', alpha=0.6)

    # plot the individual voxels as points
    # using a color map to show the progression of time
    colors = np.linspace(0, 1, len(path))
    scatter = ax.scatter(x_coords, z_coords, y_coords, c=colors, cmap='viridis', s=20)
    
    # add colorbar to indicate time step
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.5, aspect=10)
    cbar.set_label('Time Progression (Start -> End)')

    # mark start and end points
    ax.scatter(x_coords[0], z_coords[0], y_coords[0], color='green', s=100, label='Start', marker='o')
    ax.scatter(x_coords[-1], z_coords[-1], y_coords[-1], color='red', s=100, label='End', marker='x')

    # set axis limits based on your defined DIMs
    ax.set_xlim(0, x_dim)
    ax.set_ylim(0, z_dim)
    ax.set_zlim(0, y_dim)

    # labels
    ax.set_xlabel('X Dimension')
    ax.set_ylabel('Z Dimension')
    ax.set_zlabel('Y Dimension')
    ax.set_title('3D Target Trajectory')
    ax.legend()

    plt.show()

print("Generating animation frames...")
# create animation object
# interval=50 -> 50ms time interval between frames
ani = animation.FuncAnimation(fig, update, frames=TIME_STEPS, interval=50, blit=False)

# save GIF using PillowWriter
output_filename = './images/holographic_animation/3d_target_perimeter.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)

print("Done! GIF saved.")
plt.close(fig)

## Multiple trap animation

In [ ]:
# sin wave path on y axis
# circular pattern in x/z plane
def get_corkscrew_path(steps, x_max, y_max, z_max, xz_loops=2, y_loops=1):
    """
    Calculates the (x, y, z) coordinates for a target moving in a 
    circular pattern in the x/z plane while oscillating on the y axis.
    """
    path = []
    
    x_mid = x_max / 2
    y_mid = y_max / 2
    z_mid = z_max / 2

    x_radius = (x_max / 2) - 1
    y_radius = (y_max / 2) - 1
    z_radius = (z_max / 2) - 1

    for t in range(steps):
        fraction = t / steps
        
        theta = fraction * xz_loops * 2 * np.pi
        x = x_mid + x_radius * np.cos(theta)
        z = z_mid + z_radius * np.sin(theta)

        phi = fraction * y_loops * 2 * np.pi
        y = y_mid - y_radius * np.cos(phi)
            
        # floor the float and make sure its in bounds
        x_val = max(0, min(x_max - 1, int(x)))
        y_val = max(0, min(y_max - 1, int(y)))
        z_val = max(0, min(z_max - 1, int(z)))

        path.append((x_val, y_val, z_val))
        
    return path


In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# set fixed axis limits - stop potential resizing during target travel
ax.set_xlim(0, X_DIM)
ax.set_ylim(0, Z_DIM)
ax.set_zlim(0, Y_DIM)

# add labels
ax.set_xlabel('X Voxel')
ax.set_ylabel('Z Voxel')
ax.set_zlabel('Y Voxel (Height)')
ax.set_title('3D Target Movement (GIF Generation)')

# draw wireframe at the midpoint to show the path's plane
xx, zz = np.meshgrid([0, X_DIM], [0, Z_DIM])
yy = np.full_like(xx, Y_DIM // 2)
ax.plot_wireframe(xx, zz, yy, color='gray', alpha=0.2)

# create a color map to improve y axis visualization
cmap = cm.viridis
norm = mcolors.Normalize(vmin=0, vmax=Y_DIM)

# initialize the plot element that will move (the target voxel)
# we start with empty data []; 
# s (size)=150, c (color)='red' is color
target_one = ax.scatter([], [], [], s=150, marker='o', edgecolors='black')
target_two = ax.scatter([], [], [], s=150, marker='o', edgecolors='black')

# add color bar as legend for y axis value
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Y-Axis Height')


In [ ]:
def get_multitarget_paths():
    xy_path = get_target_path(TIME_STEPS, X_DIM, Y_DIM, Z_DIM)
    corkscrew_path = get_corkscrew_path(TIME_STEPS, X_DIM, Y_DIM, Z_DIM)

    return [xy_path, corkscrew_path]

multi_trap = get_multitarget_paths()

def update_2trap(frame):
    """
    This function is called for every frame of the animation.
    'frame' is an integer counting from 0 to frames-1.
    """
    # get coordinate for the current time step
    x1, y1, z1 = multi_trap[0][frame]
    x2, y2, z2 = multi_trap[1][frame]
    
    # update the 3D scatter plot position.
    # matplotlib 3D scatter requires updating internal _offsets3d
    # wrap x, y, z in lists because scatter expects sequences
    target_one._offsets3d = ([x1], [z1], [y1])
    target_two._offsets3d = ([x2], [z2], [y2])

    # set colors according to height for each target
    target_one_color = cmap(norm(y1))
    target_two_color = cmap(norm(y2))

    target_one.set_color(target_one_color)
    target_two.set_color(target_two_color)
    
    # update title for each time step
    ax.set_title(f'Target Movement - Time Step: {frame}/{TIME_STEPS}')
    
    # return updated display
    return target_scatter,

In [ ]:
ani = animation.FuncAnimation(fig, update_2trap, frames=TIME_STEPS, interval=50, blit=False)

# save GIF using PillowWriter
output_filename = './images/holographic_animation/multitarget.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)

print("Done! GIF saved.")
plt.close(fig)

### Use slmsuite to find phase masks to generate multiplane (3D) traps moving along paths described above

In [ ]:
from slmsuite.holography.algorithms import CompressedSpotHologram
from slmsuite.hardware.slms.simulated import SimulatedSLM
from slmsuite.hardware.cameras.simulated import SimulatedCamera
from slmsuite.hardware.cameraslms import FourierSLM
from slmsuite.holography.toolbox import convert_vector

# removing warnings from slmsuite for divide by 0 as this is already
# handled by the library.
warnings.filterwarnings("ignore", category=UserWarning, module="slmsuite")
warnings.filterwarnings("ignore", category=RuntimeWarning)


multi_trap = np.array(multi_trap)

# simple fix for raised ValueError because traps too close to edge
slm_shape = (Y_DIM*2, X_DIM*2)

# using simulated 532 nm laser
wav_um = 0.532

slm = SimulatedSLM((slm_shape[1], slm_shape[0]), pitch_um=(5, 5), wav_um=wav_um)
cam = SimulatedCamera(slm, pitch_um=(5, 5))
fs = FourierSLM(cam, slm)
fs.fourier_calibrate(plot=False)

# store list of precalculated phase masks for animation
phase_masks = []

for t in tqdm(range(TIME_STEPS), desc="Optimizing Holograms"):
    # extract coordinates for current time step t
    # dim 1: which of the two traps
    # dim 2: time step we are calculating phase mask for
    # dim 3: x,y,z tuple
    trap_locs = multi_trap[:, t, :].T

    trap_locs_ij = trap_locs[:2, :]
    trap_locs_z = trap_locs[2, :]

    # translate i and j coordinates by 128 pixels to avoid integration width moving outside
    # expected calculatable region, avoiding ValueError
    trap_locs_ij = trap_locs_ij + np.array([[128], [128]])

    trap_locs_focal_power = trap_locs_z * Z_SCALE

    trap_locs_kxy = convert_vector(
        trap_locs_ij,
        from_units="ij",
        to_units="kxy",
        hardware=fs
    )

    trap_locs_kxyz = np.vstack((trap_locs_kxy, trap_locs_focal_power))

    trap_locs_zernike = convert_vector(
        trap_locs_kxyz,
        from_units="kxy",
        to_units="zernike",
        hardware=fs
    )

    # Initialize a CompressedSpotHologram
    # basis="zernike" for D==3 means basis is [2,1,4] 
    # basis vector correspond to x/y tilt and focus term for zernike polynomials (2nd, 1st, 4th)
    # Zernike Polynomials: https://en.wikipedia.org/wiki/Zernike_polynomials
    hologram = CompressedSpotHologram(
        spot_vectors=trap_locs_zernike,
        basis=[2,1,4],
        cameraslm=fs
    )

    hologram.optimize(method="WGS-Leonardo", maxiter=20)

    phase_masks.append(hologram.get_phase())

phase_video = np.array(phase_masks)

In [ ]:
# Create gif of phase mask
fig, ax = plt.subplots(figsize=(6, 6))
plt.axis('off')

im = ax.imshow(phase_video[0], cmap='twilight')
title = ax.set_title("Phase Mask - Time Step: 0")

def update_phase(frame):
    im.set_array(phase_video[frame])
    title.set_text(f"Phase Mask - Time Step: {frame}/{TIME_STEPS}")
    return [im, title]

ani = animation.FuncAnimation(fig, update_phase, frames=TIME_STEPS, interval=50, blit=False)

output_filename = './images/holographic_animation/multitarget_phasemasks.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)

print("Done! GIF saved.")
plt.close(fig)

In [ ]:
# obtain camera capture
farfield_frames = []

for phase_mask in tqdm(phase_video, desc="Simulating Camera"):
    # write current phase mask to simulated slm
    slm.write(phase_mask)
    
    cam.flush()
    img = cam.get_image()
    
    farfield_frames.append(img)

farfield_video = np.array(farfield_frames)

In [ ]:
# create animation of holographic optical trap reconstruction from calculated phase masks
fig, ax = plt.subplots(figsize=(6, 6))
plt.axis('off')

vmax = np.percentile(farfield_video, 99.9) 
im = ax.imshow(farfield_video[0], cmap='inferno', vmax=vmax)
title = ax.set_title("Reconstructed Traps - Time Step: 0")

def update_farfield(frame):
    im.set_array(farfield_video[frame])
    title.set_text(f"Reconstructed Traps - Time Step: {frame}/{TIME_STEPS}")
    return [im, title]

ani = animation.FuncAnimation(fig, update_farfield, frames=TIME_STEPS, interval=50, blit=False)

output_filename = './images/holographic_animation/reconstructed_traps.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)

print("Done! Farfield GIF saved.")
plt.close(fig)

In [ ]:
# combine phase mask and reconstruction animations for presentation

# create 2 column figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.axis('off')
ax2.axis('off')

im1 = ax1.imshow(phase_video[0], cmap='twilight')
title1 = ax1.set_title("Phase Mask (SLM) - Frame: 0")

vmax = np.percentile(farfield_video, 99.9)
im2 = ax2.imshow(farfield_video[0], cmap='inferno', vmax=vmax)
title2 = ax2.set_title("Reconstructed Traps - Frame: 0")

# create frame update function
def update_phasemask_and_reconstruction(frame):
    im1.set_array(phase_video[frame])
    im2.set_array(farfield_video[frame])
    
    title1.set_text(f"Phase Mask (SLM) - Frame: {frame}/{TIME_STEPS}")
    title2.set_text(f"Reconstructed Traps - Frame: {frame}/{TIME_STEPS}")
    
    return [im1, im2, title1, title2]

ani = animation.FuncAnimation(fig, update_phasemask_and_reconstruction, frames=TIME_STEPS, interval=50, blit=False)

# save gif
output_filename = './images/holographic_animation/phasemask_and_reconstruction.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)

print("Done! Side-by-side GIF saved.")
plt.close(fig)

In [ ]:
# create 3 column combined graph for presentation
# displays original/intended traps, calculated phase masks, and reconstructed traps

fig = plt.figure(figsize=(18, 6))

# subfigures: 3d scatter plot, phase mask, reconstructed traps
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# get first frame coordinates
x1_0, y1_0, z1_0 = multi_trap[0][0]
x2_0, y2_0, z2_0 = multi_trap[1][0]

# setup first frame scatter objects
target_one = ax1.scatter([x1_0], [z1_0], [y1_0], color=cmap(norm(y1_0)), s=50)
target_two = ax1.scatter([x2_0], [z2_0], [y2_0], color=cmap(norm(y2_0)), s=50)

# set dimensions for figure, swap y/z for matplotlib
ax1.set_xlim(0, X_DIM)
ax1.set_ylim(0, Z_DIM)
ax1.set_zlim(0, Y_DIM)
title1 = ax1.set_title(f"3D Target Path - Frame: 0/{TIME_STEPS}")

# setup phase mask first frame
ax2.axis('off')
im2 = ax2.imshow(phase_video[0], cmap='twilight')
title2 = ax2.set_title("Phase Mask (SLM)")

# setup reconstruction first frame
ax3.axis('off')
vmax = np.percentile(farfield_video, 99.9)
im3 = ax3.imshow(farfield_video[0], cmap='inferno', vmax=vmax)
title3 = ax3.set_title("Reconstructed Traps (Camera)")

# create frame update function
def update_all(frame):
    x1, y1, z1 = multi_trap[0][frame]
    x2, y2, z2 = multi_trap[1][frame]
    
    target_one._offsets3d = ([x1], [z1], [y1])
    target_two._offsets3d = ([x2], [z2], [y2])

    target_one.set_color(cmap(norm(y1)))
    target_two.set_color(cmap(norm(y2)))
    title1.set_text(f'3D Target Path - Frame: {frame}/{TIME_STEPS}')
    
    im2.set_array(phase_video[frame])
    im3.set_array(farfield_video[frame])
    
    return [target_one, target_two, title1, im2, im3]

# create gif and provide output
print("Generating GIF...")
ani = animation.FuncAnimation(fig, update_all, frames=TIME_STEPS, interval=50, blit=False)

output_filename = './images/holographic_animation/final_holography_calculation_display.gif'
print(f"Saving to {output_filename} (this might take a moment)...")
writer = animation.PillowWriter(fps=20)
ani.save(output_filename, writer=writer)